---
title: Solving LinkedIn Patches by Linear Optimization
date: 2026-04-10
---

![Patches Banner](../assets/5-patches/banner.jpg)

Patches is a LinkedIn minigame where the goal is to partition the board into labeled rectangular patches. This article shows how to model Patches as a Binary Linear Optimization Problem using Pyomo and visualize solutions with NetworkX and Matplotlib.

# How to Play

:::{figure} ../assets/5-patches/example.mp4
:label: patches-example
:alt: Patches Example
:align: center
:width: 50%
Example of a Patches minigame board. _Source: [LinkedIn](https://www.linkedin.com/games/patches/)_
:::

Objective
: Partition the board into non-overlapping rectangular patches so that each patch meets the tip prescriptions on their tip squares.

Rules
: All rectangles have a tip square that may indicates the rectangle's type (vertical, horizontal, square, or any rectangle format) and/or its area, which must be respected to solve the game;
: The tip squares must be covered by a rectangle that attends its prescriptions;
: A rectangle must cover one and only one tip square;
: The area of ​​all rectangles must be greater than 1 square on the board. That is, rectangles with dimensions 1 $\times$ 1 are not allowed.

---


---

# Problem Modeling

## Ranges

$I = \{1, \cdots, m\}$
: The row range, where $m$ is the total amount of rows on game board.

$J = \{1, \cdots, n\}$
: The column range, where $n$ is the total amount of columns on game board.

$K = \{1, \cdots, p\}$
: Range of all rectangles to be drawn on board game, where $p$ is the total of rectangles to be drawn on board.

## Sets

$S = I \times J = \{(i,j) \mid \forall i \in I, \forall j \in J\}$
: Set of all board squares, that is just the Cartesian product between sets $I$ and $J$.

$T = \{(i,j,k) \mid  i \in I, j \in J, k \in K\} \subset I \times J \times K$
: Set of board squares $(i,j)$ with tips about a rectangle $k$.

$V \subseteq K$
: Set of vertical rectangles, i. e., rectangles whose height is greater than its width.

$H \subseteq K$
: Set of horizontal rectangles, i. e., rectangles whose width is greater than its height.

$Q \subseteq K$
: Set of square figures, i. e., rectangles whose width is equal to its height.

$A \subseteq K$
: Set of rectangles with a predefined area $a_k$.

## Decision Variables

$x_{ijk} \in \{0,1\}, \forall (i,j,k) \in I \times J \times K$
: $x_{ijk} = 1$, if the square $(i,j)$ will be covered by rectangle $k$.
: $x_{ijk} = 0$, otherwise

$c_k \in I, \forall k \in K$
: The column index of the upper leftmost cell of the rectangle $k$.

$r_k \in J, \forall k \in K$
: The row index of the upper leftmost cell of the rectangle $k$.

$w_k \in I, \forall k \in K$
: Width of the rectangle $k$.

$h_k \in J, \forall k \in K$
: Height of the rectangle $k$.

## Parameters

$a_k \in \mathbb{N}$
: Predefined area for the rectangle $k \in A$.

## Objective function

$$\text{Min} \ \sum_{k \in K}{w_k + h_k} $$

## Constraints

Domain Constraints
: Sets of constraints for the domain of decision variables

    Binarity Constraints
    : Each $x_{ijk}$ must be a binary variable
    : $$x_{ijk} \in \{0,1\}, \forall (i,j,k) \in I \times J \times K$$

    Integrity Constraints
    : The remaining decision variables must be integers.
    : $$c_k \in I, \forall k \in K$$
    : $$w_k \in I, \forall k \in K$$
    : $$r_k \in J, \forall k \in K$$
    : $$h_k \in J, \forall k \in K$$

Unique-Rectangle-Per-Square Constraints
: Each board square $(i,j)$ must be covered by only one rectangle $k$.
: $$\sum_{k \in K}{x_{ijk}} = 1, \forall (i,j) \in S$$

Continuity Constraints

    Row-Continuity Constraints
    : $$x_{ij'k} + x_{ij"k} - 1 \le x_{ijk}, \forall i \in I, \forall j' < j < j", \forall k \in K$$

    Column-Continuity Constraints
    : $$x_{i'jk} + x_{i"jk} - 1 \le x_{ijk}, \forall i' < i < i", \forall j \in J, \forall k \in K$$

Rectangle-Inside-Board Constraints
: These sets of constraints ensure that any rectangle $k$ lies within the grid's row and column boundaries.

    First-Row-Position Constraints
    : For each rectangle $k$, the position $r_k$ of its first row must be equal or greater than 1.
    : $$r_k \ge 1, \forall k \in K$$

    Last-Row-Position Constraints
    : For each rectangle $k$, the position of its last row, given by the expression $r_k + h_k - 1$, must be equal or less than the total amount of rows $n$.
    : $$r_k + h_k - 1 \le n, \forall k \in K$$

    First-Column-Position Constraints
    : For each rectangle $k$, the position $c_k$ of its first column must be equal or greater than 1.
    : $$c_k \ge 1, \forall k \in K$$

    Last-Column-Position Constraints
    : For each rectangle $k$, the position of its last column, given by the expression $c_k + w_k - 1$, must be equal or less than the total amount of columns $m$.
    : $$c_k + w_k - 1 \le m, \forall k \in K$$

Square-Inside-Rectangle Constraints
: These sets of constraints ensure that any square $(i,j)$ covered by a rectangle $k$ lies within its row and column boundaries.

    Row-Lower-Bound-Coverage Constraints
    : if a square $(i,j)$ is covered by rectangle $k$, then its row index $i$ must be equal or greater than its first row's index $r_k$.
    : $$r_k - i \le M(1-x_{ijk}), \forall (i,j,k) \in I \times J \times K$$

    Row-Upper-Bound-Coverage Constraints
    : if a square $(i,j)$ is covered by rectangle $k$, then its row index $i$ must be equal or less than its last row's index, given by $r_k + h_k - 1$.
    : $$i - (r_k + h_k - 1) \le M(1-x_{ijk}), \forall (i,j,k) \in I \times J \times K$$

    Column-Lower-Bound-Coverage Constraints
    : if a square $(i,j)$ is covered by rectangle $k$, then its column index $j$ must be equal or greater than its first column's index $c_k$.
    : $$c_k - j \le M(1-x_{ijk}), \forall (i,j,k) \in I \times J \times K$$

    Column-Upper-Bound-Coverage Constraints
    : if a square $(i,j)$ is covered by rectangle $k$, then its column index $j$ must be equal or less its last column's index, given by $c_k + w_k - 1$.
    : $$j - (c_k + w_k - 1) \le M(1-x_{ijk}), \forall (i,j,k) \in I \times J \times K$$

Tip Constraints
: These sets of constraints deal with prescriptions found on the tip squares.

    Tip-Square Constraints
    : For each square $(i,j)$ with tips about rectangle $k$, it must be covered by $k$.
    : $$x_{ijk} = 1, \forall (i,j,k) \in T$$

    Prescribed-Area Constraints
    : For each rectangle $k$ with a predefined area $a_k$, the sum of its covered squares $x_{ijk}$ must be equal to $a_k$
    : $$\sum_{(i,j) \in S}{x_{ijk}} = a_k, \forall k \in A$$

    Vertical-Rectangle Constraints
    : For each vertical rectangle $k$, its height $h_k$ must be greater than its width $w_k$.
    : $$w_k < h_k, \forall k \in V$$

    Horizontal-Rectangle Constraints
    : For each horizontal rectangle $k$, its width $w_k$ must be greater than its height $h_k$.
    : $$w_k > h_k, \forall k \in H$$

    Square-Rectangle Constraints
    : For each square rectangle $k$, its width $w_k$ must be equal to its height $h_k$.
    : $$w_k = h_k, \forall k \in Q$$

::: {literalinclude} ../../linkedin-games/patches/patches.py
:language: python
:::


# Solving with Pyomo and NetworkX

The code examples in this book use Pyomo for modeling and NetworkX/Matplotlib for visualization. Ensure a solver is available in the environment. The Patches implementation uses the HiGHS solver (open-source). To install it:

:::{code} bash
pip install highspy
:::


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pyomo.environ as pyo
from linkedin_games.patches import Patches, Rectangle, RecType

# Display versions (optional)
print('pyomo', pyo.__version__)
print('networkx', nx.__version__)
print('matplotlib', plt.matplotlib.__version__)

# Concrete instance and solution

:::{figure} ../assets/5-patches/problem.jpg
:label: patches-problem
:alt: Patches problem instance
:align: center
:width: 50%
Patches problem instance used in this article. _Source: LinkedIn_
:::

The following cell builds the rectangles dictionary for the concrete instance and solves it using the Patches model.


In [ ]:
# Example rectangles (colors and tips taken from the repo example)
rectangles = {
    'yellow':  Rectangle((1, 2), RecType.ANY,      2, '#846A0B'),
    'teal':    Rectangle((1, 4), RecType.ANY,      6, '#096B78'),
    'purple':  Rectangle((2, 6), RecType.ANY,      2, '#5A3DB1'),
    'green':   Rectangle((3, 1), RecType.ANY,      6, '#0A7541'),
    'orange':  Rectangle((3, 3), RecType.VERTICAL, 2, '#EF6C00'),
    'red':     Rectangle((4, 4), RecType.SQUARE,   4, '#E30102'),
    'blue':    Rectangle((4, 6), RecType.ANY,      2, '#097BB1'),
    'magenta': Rectangle((5, 1), RecType.ANY,      2, '#A01E4E'),
    'brick':   Rectangle((6, 3), RecType.ANY,      6, '#9B3C1C'),
    'brown':   Rectangle((6, 5), RecType.ANY,      4, '#503B36')
}

patches = Patches((6,6), rectangles)
patches.solve()
patches.show()

:::{figure} ../assets/5-patches/solution.jpg
:label: patches-solution
:alt: Patches solution
:align: center
:width: 50%
Solved instance visualization.
:::